In [ ]:
import os
import re
import gc
import json
import time
import copy
import math
import random
import logging
import warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter
from math import pi

import numpy as np
import pandas as pd
import boto3
import requests
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.pipeline import Pipeline

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False
})

print(f'torch  {torch.__version__} | cuda: {torch.cuda.is_available()}')
print(f'pandas {pd.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print(f'seaborn {sns.__version__}')

In [ ]:
BASE_DIR    = Path(r'')
DATA_DIR    = BASE_DIR / 'Data'
RESULTS_DIR = BASE_DIR / 'Results'
LOGS_DIR    = BASE_DIR / 'Logs'
FIGURES_DIR = RESULTS_DIR / 'Figures' / 'Model_Evaluation'
CKPT_DIR    = RESULTS_DIR / 'eval_checkpoints'
FT_DIR      = RESULTS_DIR / 'finetuned_models'
TABLES_DIR  = RESULTS_DIR / 'Tables'

for d in [RESULTS_DIR, LOGS_DIR, FIGURES_DIR, CKPT_DIR, FT_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

JACV_PATH    = DATA_DIR / 'dataset_jacv.json'
RESULTS_JSON = RESULTS_DIR / 'model_evaluation_results.json'
LOG_PATH     = LOGS_DIR / 'model_evaluation.log'
ENV_PATH     = BASE_DIR / '.env'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(LOG_PATH, mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('JACV_Eval')
logger.info('Notebook 04_Model_Evaluation STARTED')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = bool(torch.cuda.is_available())

MAX_RETRIES   = 3
API_SLEEP     = 0.5
CV_SPLITS     = 3
MAX_LENGTH    = 512     
TRAIN_BS      = 2
EVAL_BS       = 4        
GRAD_ACCUM    = 4
VAL_SIZE      = 0.12
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
FT_EPOCHS     = 3
EMBED_BATCH   = 4

DEVICE_NAME = str(DEVICE)
print(f'Device: {DEVICE_NAME}')
print(f'MAX_LENGTH={MAX_LENGTH} | TRAIN_BS={TRAIN_BS} | EVAL_BS={EVAL_BS} | GRAD_ACCUM={GRAD_ACCUM}')

In [ ]:
def load_env(path):
    env = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                env[k.strip()] = v.strip()
    return env

ENV = load_env(ENV_PATH)
print(f'Env loaded: {list(ENV.keys())}')

bedrock = boto3.client(
    'bedrock-runtime',
    region_name=ENV['AWS_DEFAULT_REGION'],
    aws_access_key_id=ENV['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=ENV['AWS_SECRET_ACCESS_KEY'],
)

MARITACA_KEY   = ENV.get('MARITACA_KEY', '')
MARITACA_MODEL = ENV.get('MARITACA_MODEL', 'sabia-4')

BEDROCK_MODELS = {
    'Claude-Sonnet-4.6': 'global.anthropic.claude-sonnet-4-6',
    'Claude-Opus-4.7': 'global.anthropic.claude-opus-4-7',
    'Mistral-Large-3': 'mistral.mistral-large-3-675b-instruct'
}

print(f'Maritaca model : {MARITACA_MODEL}')
print(f'Bedrock models : {list(BEDROCK_MODELS.keys())}')

In [ ]:
logger.info('Loading JACV dataset...')
with open(JACV_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

instances_cls = [d for d in dataset if d['task'] == 'JACV-CLS']
instances_adv = [d for d in dataset if d['task'] == 'JACV-ADV']

print(f'JACV-CLS instances : {len(instances_cls)}')
print(f'JACV-ADV instances : {len(instances_adv)}')

label_dist = Counter(d['gold_label'] for d in instances_cls)
print(f'CLS label dist : {dict(label_dist)}')
logger.info(f'CLS={len(instances_cls)} ADV={len(instances_adv)} Labels={dict(label_dist)}')

CLS_LABELS = ['COHERENT', 'INCOHERENT', 'CONTRADICTORY']
label2id = {lbl: i for i, lbl in enumerate(CLS_LABELS)}
id2label = {i: lbl for lbl, i in label2id.items()}

def trunc(text, max_words=500):
    return ' '.join(str(text).split()[:max_words])

cls_record_ids = np.array([d['record_id'] for d in instances_cls])
adv_record_ids = np.array([d['record_id'] for d in instances_adv])

y_cls_gold = np.array([d['gold_label'] for d in instances_cls])
y_adv_gold = np.array([d['coherent_option'] for d in instances_adv])

y_cls_gold_ids = np.array([label2id[y] for y in y_cls_gold])

cls_pair_texts = [
    trunc(d['direito'], 320) + ' [SEP] ' + trunc(d['pedido'], 220)
    for d in instances_cls
]

adv_pair_texts_a = [
    trunc(d['direito'], 320) + ' [SEP] ' + trunc(d['pedido_A'], 220)
    for d in instances_adv
]
adv_pair_texts_b = [
    trunc(d['direito'], 320) + ' [SEP] ' + trunc(d['pedido_B'], 220)
    for d in instances_adv
]

gkf = GroupKFold(n_splits=CV_SPLITS)
CV_FOLDS = list(gkf.split(np.arange(len(instances_cls)), y_cls_gold_ids, groups=cls_record_ids))

print(f'Prepared {len(CV_FOLDS)} grouped folds using record_id.')

In [ ]:
def prompt_cls(direito, pedido):
    return '\n'.join([
        'Tarefa: Avalie a coerência lógica entre a fundamentação jurídica e o pedido em um documento judicial brasileiro.',
        '',
        '--- FUNDAMENTAÇÃO JURÍDICA (DIREITO) ---',
        trunc(direito, 500),
        '--- PEDIDO ---',
        trunc(pedido, 300),
        '---',
        'Classifique o PEDIDO com UMA das opções abaixo:',
        '- COHERENT',
        '- INCOHERENT',
        '- CONTRADICTORY',
        '',
        'Responda APENAS com a palavra de classificação.'
    ])

def prompt_adv(direito, pedido_a, pedido_b):
    return '\n'.join([
        'Tarefa: Qual dos pedidos abaixo tem coerência lógica com a fundamentação jurídica?',
        '',
        '--- FUNDAMENTAÇÃO JURÍDICA (DIREITO) ---',
        trunc(direito, 500),
        '--- PEDIDO A ---',
        trunc(pedido_a, 300),
        '--- PEDIDO B ---',
        trunc(pedido_b, 300),
        '---',
        'Qual pedido é COERENTE com a fundamentação? (Responda apenas A ou B)'
    ])

def parse_cls(text):
    t = str(text).upper().strip()
    if 'INCOHERENT' in t or 'INCOEREN' in t:
        return 'INCOHERENT'
    if 'COHERENT' in t or 'COEREN' in t:
        return 'COHERENT'
    if 'CONTRADICTORY' in t or 'CONTRAD' in t:
        return 'CONTRADICTORY'
    return 'ERROR'

def parse_adv(text):
    t = str(text).upper().strip()
    if t.startswith('A'):
        return 'A'
    if t.startswith('B'):
        return 'B'
    if 'PEDIDO A' in t or '"A"' in t:
        return 'A'
    if 'PEDIDO B' in t or '"B"' in t:
        return 'B'
    return 'ERROR'

def ckpt_path(model_key):
    safe = model_key.replace('/', '_').replace(' ', '_')
    return CKPT_DIR / f'ckpt_{safe}.json'

def save_prediction_checkpoint(model_key, cls_preds, adv_preds, cls_raw=None, adv_raw=None):
    ckpt = {'cls': {}, 'adv': {}}

    for inst, pred in zip(instances_cls, cls_preds):
        item = {'gold': inst['gold_label'], 'pred': pred}
        if cls_raw is not None:
            item['raw'] = cls_raw.get(inst['jacv_id'], '')
        ckpt['cls'][inst['jacv_id']] = item

    for inst, pred in zip(instances_adv, adv_preds):
        item = {'gold': inst['coherent_option'], 'pred': pred}
        if adv_raw is not None:
            item['raw'] = adv_raw.get(inst['jacv_id'], '')
        ckpt['adv'][inst['jacv_id']] = item

    with open(ckpt_path(model_key), 'w', encoding='utf-8') as f:
        json.dump(ckpt, f, indent=2, ensure_ascii=False)

def load_prediction_checkpoint(model_key):
    p = ckpt_path(model_key)
    if not p.exists():
        return None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f)

def bootstrap_ci_accuracy(y_true, y_pred, n_boot=2000, seed=SEED):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rng = np.random.default_rng(seed)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(accuracy_score(y_true[idx], y_pred[idx]))
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return float(lo), float(hi)

def bootstrap_ci_f1macro(y_true, y_pred, n_boot=2000, seed=SEED):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rng = np.random.default_rng(seed)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0))
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return float(lo), float(hi)

def make_result_bundle(y_cls_true, y_cls_pred, y_adv_true, y_adv_pred, model_type):
    cls_acc = accuracy_score(y_cls_true, y_cls_pred)
    cls_f1  = f1_score(y_cls_true, y_cls_pred, average='macro', zero_division=0)
    adv_acc = accuracy_score(y_adv_true, y_adv_pred)

    return {
        'cls_acc': float(cls_acc),
        'cls_f1_macro': float(cls_f1),
        'adv_acc': float(adv_acc),
        'cls_acc_ci95': bootstrap_ci_accuracy(y_cls_true, y_cls_pred),
        'cls_f1_ci95': bootstrap_ci_f1macro(y_cls_true, y_cls_pred),
        'adv_acc_ci95': bootstrap_ci_accuracy(y_adv_true, y_adv_pred),
        'type': model_type
    }

def result_from_ckpt(model_key, model_type):
    ckpt = load_prediction_checkpoint(model_key)
    if ckpt is None:
        return None

    y_c = [v['gold'] for v in ckpt['cls'].values()]
    p_c = [v['pred'] for v in ckpt['cls'].values()]
    y_a = [v['gold'] for v in ckpt['adv'].values()]
    p_a = [v['pred'] for v in ckpt['adv'].values()]

    result = make_result_bundle(y_c, p_c, y_a, p_a, model_type)
    result['cls_errors'] = p_c.count('ERROR')
    result['adv_errors'] = p_a.count('ERROR')
    return result

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('Prompt / parser / checkpoint / metrics helpers ready.')

In [ ]:
def load_or_init_ckpt(model_key):
    p = ckpt_path(model_key)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'cls': {}, 'adv': {}}

def flush_ckpt(model_key, ckpt):
    with open(ckpt_path(model_key), 'w', encoding='utf-8') as f:
        json.dump(ckpt, f, indent=2, ensure_ascii=False)

def finalize_result_from_ckpt(model_key, model_type):
    ckpt = load_prediction_checkpoint(model_key)
    if ckpt is None:
        return None

    y_c = [v['gold'] for v in ckpt.get('cls', {}).values()]
    p_c = [v['pred'] for v in ckpt.get('cls', {}).values()]
    y_a = [v['gold'] for v in ckpt.get('adv', {}).values()]
    p_a = [v['pred'] for v in ckpt.get('adv', {}).values()]

    result = {
        'type': model_type,
        'cls_acc': np.nan,
        'cls_f1_macro': np.nan,
        'adv_acc': np.nan,
    }

    if y_c and p_c:
        result['cls_acc'] = float(accuracy_score(y_c, p_c))
        result['cls_f1_macro'] = float(f1_score(y_c, p_c, average='macro', zero_division=0))
        result['cls_acc_ci95'] = bootstrap_ci_accuracy(y_c, p_c)
        result['cls_f1_ci95'] = bootstrap_ci_f1macro(y_c, p_c)
        result['cls_errors'] = p_c.count('ERROR')

    if y_a and p_a:
        result['adv_acc'] = float(accuracy_score(y_a, p_a))
        result['adv_acc_ci95'] = bootstrap_ci_accuracy(y_a, p_a)
        result['adv_errors'] = p_a.count('ERROR')

    return result

In [ ]:
RESULTS = {'metadata': {'started_at': datetime.now().isoformat()}, 'models': {}}

majority_cls = Counter(y_cls_gold).most_common(1)[0][0]

random.seed(SEED)
y_random_cls = [random.choice(CLS_LABELS) for _ in y_cls_gold]
y_random_adv = [random.choice(['A', 'B']) for _ in y_adv_gold]
y_majority_cls = [majority_cls] * len(y_cls_gold)
y_majority_adv = ['A'] * len(y_adv_gold)  

RESULTS['models']['Random-Baseline'] = make_result_bundle(
    y_cls_gold, y_random_cls, y_adv_gold, y_random_adv, 'statistical'
)
RESULTS['models']['Majority-Baseline'] = make_result_bundle(
    y_cls_gold, y_majority_cls, y_adv_gold, y_majority_adv, 'statistical'
)

save_prediction_checkpoint('Random-Baseline', y_random_cls, y_random_adv)
save_prediction_checkpoint('Majority-Baseline', y_majority_cls, y_majority_adv)

cached = result_from_ckpt('TF-IDF-LogReg', 'statistical')
if cached is not None:
    RESULTS['models']['TF-IDF-LogReg'] = cached
    print('Loaded TF-IDF-LogReg from checkpoint.')
else:
    print('Running TF-IDF + LogReg with grouped CV...')
    oof_cls = ['ERROR'] * len(instances_cls)
    oof_adv = ['ERROR'] * len(instances_adv)

    for fold, (train_idx, test_idx) in enumerate(CV_FOLDS, 1):
        logger.info(f'TF-IDF fold {fold}/{CV_SPLITS}')

        X_train = [cls_pair_texts[i] for i in train_idx]
        y_train = [y_cls_gold[i] for i in train_idx]

        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)),
            ('clf', LogisticRegression(
                max_iter=1500,
                C=1.0,
                class_weight='balanced',
                random_state=SEED
            ))
        ])

        pipe.fit(X_train, y_train)

        X_test = [cls_pair_texts[i] for i in test_idx]
        pred_cls = pipe.predict(X_test)
        for idx_local, pred in zip(test_idx, pred_cls):
            oof_cls[idx_local] = pred

        test_record_set = set(cls_record_ids[test_idx])
        adv_test_idx = [j for j, rid in enumerate(adv_record_ids) if rid in test_record_set]

        coh_idx = list(pipe.named_steps['clf'].classes_).index('COHERENT')
        proba_a = pipe.predict_proba([adv_pair_texts_a[j] for j in adv_test_idx])[:, coh_idx]
        proba_b = pipe.predict_proba([adv_pair_texts_b[j] for j in adv_test_idx])[:, coh_idx]
        pred_adv = ['A' if proba_a[k] >= proba_b[k] else 'B' for k in range(len(adv_test_idx))]

        for idx_local, pred in zip(adv_test_idx, pred_adv):
            oof_adv[idx_local] = pred

    RESULTS['models']['TF-IDF-LogReg'] = make_result_bundle(
        y_cls_gold, oof_cls, y_adv_gold, oof_adv, 'statistical'
    )
    save_prediction_checkpoint('TF-IDF-LogReg', oof_cls, oof_adv)

print('Statistical baselines done.')
for m in ['Random-Baseline', 'Majority-Baseline', 'TF-IDF-LogReg']:
    r = RESULTS['models'][m]
    print(f'  {m:25s} CLS={r["cls_acc"]:.3f} ADV={r["adv_acc"]:.3f}')

In [ ]:
def mean_pool_embeddings(texts, tokenizer, model, batch_size=EMBED_BATCH, max_length=MAX_LENGTH):
    all_embs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)

        last_hidden = out.last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = F.normalize(pooled, dim=1)

        all_embs.append(pooled.cpu().numpy())

    return np.vstack(all_embs)

def eval_embedding_logreg_groupcv(model_id, model_key):
    cached = result_from_ckpt(model_key, 'embedding_supervised')
    if cached is not None:
        print(f'Loaded {model_key} from checkpoint.')
        return cached

    print(f'\nLoading embedding model: {model_key}')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(DEVICE).eval()

    print(f'Encoding CLS pairs for {model_key}...')
    emb_cls = mean_pool_embeddings(cls_pair_texts, tokenizer, model)

    print(f'Encoding ADV-A pairs for {model_key}...')
    emb_adv_a = mean_pool_embeddings(adv_pair_texts_a, tokenizer, model)

    print(f'Encoding ADV-B pairs for {model_key}...')
    emb_adv_b = mean_pool_embeddings(adv_pair_texts_b, tokenizer, model)

    oof_cls = ['ERROR'] * len(instances_cls)
    oof_adv = ['ERROR'] * len(instances_adv)

    for fold, (train_idx, test_idx) in enumerate(CV_FOLDS, 1):
        logger.info(f'{model_key} fold {fold}/{CV_SPLITS}')

        clf = LogisticRegression(
            max_iter=1500,
            class_weight='balanced',
            random_state=SEED
        )

        clf.fit(emb_cls[train_idx], y_cls_gold[train_idx])

        pred_cls = clf.predict(emb_cls[test_idx])
        for idx_local, pred in zip(test_idx, pred_cls):
            oof_cls[idx_local] = pred

        test_record_set = set(cls_record_ids[test_idx])
        adv_test_idx = [j for j, rid in enumerate(adv_record_ids) if rid in test_record_set]

        coh_idx = list(clf.classes_).index('COHERENT')
        proba_a = clf.predict_proba(emb_adv_a[adv_test_idx])[:, coh_idx]
        proba_b = clf.predict_proba(emb_adv_b[adv_test_idx])[:, coh_idx]
        pred_adv = ['A' if proba_a[k] >= proba_b[k] else 'B' for k in range(len(adv_test_idx))]

        for idx_local, pred in zip(adv_test_idx, pred_adv):
            oof_adv[idx_local] = pred

    result = make_result_bundle(y_cls_gold, oof_cls, y_adv_gold, oof_adv, 'embedding_supervised')
    save_prediction_checkpoint(model_key, oof_cls, oof_adv)

    del model
    clear_memory()
    return result

EMBED_MODELS = {
    'GAIA': 'CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it',
    'JurisBERT': 'alfaneo/jurisbert-base-portuguese-uncased',
    'BERTimbau-base': 'neuralmind/bert-base-portuguese-cased',
    'LegalBert-pt': 'raquelsilveira/legalbertpt_fp',
}

for model_key, model_id in EMBED_MODELS.items():
    try:
        RESULTS['models'][model_key] = eval_embedding_logreg_groupcv(model_id, model_key)
        r = RESULTS['models'][model_key]
        print(f'  {model_key:25s} CLS={r["cls_acc"]:.3f} ADV={r["adv_acc"]:.3f}')
    except Exception as e:
        logger.error(f'{model_key} FAILED: {e}')
        RESULTS['models'][model_key] = {
            'cls_acc': None,
            'cls_f1_macro': None,
            'adv_acc': None,
            'error': str(e),
            'type': 'embedding_supervised'
        }

print('\nEmbedding-supervised baselines done.')

In [ ]:
class TextLabelDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {'text': self.texts[idx]}
        if self.labels is not None:
            item['label'] = int(self.labels[idx])
        return item

def build_collate_fn(tokenizer, max_length=MAX_LENGTH):
    def collate(batch):
        texts = [x['text'] for x in batch]
        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        if 'label' in batch[0]:
            enc['labels'] = torch.tensor([x['label'] for x in batch], dtype=torch.long)
        return enc
    return collate

def run_eval_pass(model, dataloader):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch.pop('labels', None)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            logits = out.logits.detach().cpu()
            all_logits.append(logits)

            if labels is not None:
                all_labels.append(labels.cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0) if all_labels else None
    return logits, labels

def predict_proba_texts(model, tokenizer, texts, batch_size=EVAL_BS, max_length=MAX_LENGTH):
    dataset = TextLabelDataset(texts, labels=None)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=build_collate_fn(tokenizer, max_length=max_length)
    )

    model.eval()
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            probs = F.softmax(out.logits, dim=-1).detach().cpu().numpy()
            all_probs.append(probs)

    return np.vstack(all_probs)

def train_one_fold(
    model_id,
    model_key,
    fold_num,
    train_texts,
    train_labels,
    val_texts,
    val_labels,
    num_epochs=FT_EPOCHS
):
    fold_dir = FT_DIR / model_key / f'fold_{fold_num}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = fold_dir / 'best_model.pt'
    history_path = fold_dir / 'history.json'

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=len(CLS_LABELS),
        id2label=id2label,
        label2id=label2id
    ).to(DEVICE)

    if hasattr(model, 'gradient_checkpointing_enable'):
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass

    train_ds = TextLabelDataset(train_texts, train_labels)
    val_ds   = TextLabelDataset(val_texts, val_labels)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BS,
        shuffle=True,
        collate_fn=build_collate_fn(tokenizer)
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BS,
        shuffle=False,
        collate_fn=build_collate_fn(tokenizer)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scaler = GradScaler(enabled=USE_AMP)

    best_val_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f'{model_key} | fold {fold_num} | epoch {epoch}', leave=False)
        for step, batch in enumerate(pbar, 1):
            labels = batch['labels'].to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}

            with autocast(enabled=USE_AMP):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            running_loss += loss.item() * GRAD_ACCUM

            if (step % GRAD_ACCUM == 0) or (step == len(train_loader)):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f'{running_loss / step:.4f}')

        val_logits, val_y = run_eval_pass(model, val_loader)
        val_pred_ids = val_logits.argmax(dim=1).numpy()
        val_true_ids = val_y.numpy()

        val_acc = accuracy_score(val_true_ids, val_pred_ids)
        val_f1  = f1_score(val_true_ids, val_pred_ids, average='macro', zero_division=0)

        epoch_rec = {
            'epoch': epoch,
            'train_loss': float(running_loss / max(1, len(train_loader))),
            'val_acc': float(val_acc),
            'val_f1_macro': float(val_f1)
        }
        history.append(epoch_rec)
        print(f'{model_key} | fold {fold_num} | epoch {epoch}: val_acc={val_acc:.4f} | val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)

    with open(history_path, 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

    if best_state is None:
        best_state = model.state_dict()
        torch.save(best_state, best_model_path)

    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    return tokenizer, model, best_model_path, history_path

def eval_finetuned_groupcv(model_id, model_key, num_epochs=FT_EPOCHS):
    cached = result_from_ckpt(model_key, 'finetuned_encoder')
    if cached is not None:
        print(f'Loaded {model_key} from checkpoint.')
        return cached

    oof_cls = ['ERROR'] * len(instances_cls)
    oof_adv = ['ERROR'] * len(instances_adv)

    for fold, (train_idx, test_idx) in enumerate(CV_FOLDS, 1):
        logger.info(f'{model_key} fold {fold}/{CV_SPLITS}')

        fold_train_texts = [cls_pair_texts[i] for i in train_idx]
        fold_train_labels = [y_cls_gold_ids[i] for i in train_idx]

        sub_train_idx, sub_val_idx = train_test_split(
            np.arange(len(fold_train_texts)),
            test_size=VAL_SIZE,
            random_state=SEED,
            stratify=fold_train_labels
        )

        train_texts = [fold_train_texts[i] for i in sub_train_idx]
        train_labels = [fold_train_labels[i] for i in sub_train_idx]
        val_texts = [fold_train_texts[i] for i in sub_val_idx]
        val_labels = [fold_train_labels[i] for i in sub_val_idx]

        tokenizer, model, best_model_path, history_path = train_one_fold(
            model_id=model_id,
            model_key=model_key,
            fold_num=fold,
            train_texts=train_texts,
            train_labels=train_labels,
            val_texts=val_texts,
            val_labels=val_labels,
            num_epochs=num_epochs
        )

        test_texts = [cls_pair_texts[i] for i in test_idx]
        probs_cls = predict_proba_texts(model, tokenizer, test_texts)
        pred_cls_ids = probs_cls.argmax(axis=1)
        pred_cls = [id2label[int(i)] for i in pred_cls_ids]

        for idx_local, pred in zip(test_idx, pred_cls):
            oof_cls[idx_local] = pred

        test_record_set = set(cls_record_ids[test_idx])
        adv_test_idx = [j for j, rid in enumerate(adv_record_ids) if rid in test_record_set]

        probs_a = predict_proba_texts(model, tokenizer, [adv_pair_texts_a[j] for j in adv_test_idx])
        probs_b = predict_proba_texts(model, tokenizer, [adv_pair_texts_b[j] for j in adv_test_idx])

        coh_idx = label2id['COHERENT']
        pred_adv = ['A' if probs_a[k, coh_idx] >= probs_b[k, coh_idx] else 'B' for k in range(len(adv_test_idx))]

        for idx_local, pred in zip(adv_test_idx, pred_adv):
            oof_adv[idx_local] = pred

        del model
        clear_memory()

    result = make_result_bundle(y_cls_gold, oof_cls, y_adv_gold, oof_adv, 'finetuned_encoder')
    save_prediction_checkpoint(model_key, oof_cls, oof_adv)
    return result

print('Fine-tuning helpers ready.')

In [ ]:
FT_MODELS = {
    'RoBERTaLexPT-base': 'eduagarcia/RoBERTaLexPT-base',
    'multilíngue BERT': 'google-bert/bert-base-multilingual-cased',
}

for model_key, model_id in FT_MODELS.items():
    try:
        print(f'\n=== Running fine-tuning for {model_key} ===')
        RESULTS['models'][model_key] = eval_finetuned_groupcv(model_id, model_key, num_epochs=FT_EPOCHS)
        r = RESULTS['models'][model_key]
        print(f'  {model_key:25s} CLS={r["cls_acc"]:.3f} ADV={r["adv_acc"]:.3f}')
    except RuntimeError as e:
        logger.error(f'{model_key} runtime failure: {e}')
        RESULTS['models'][model_key] = {
            'cls_acc': None,
            'cls_f1_macro': None,
            'adv_acc': None,
            'error': str(e),
            'type': 'finetuned_encoder'
        }
        clear_memory()
    except Exception as e:
        logger.error(f'{model_key} failed: {e}')
        RESULTS['models'][model_key] = {
            'cls_acc': None,
            'cls_f1_macro': None,
            'adv_acc': None,
            'error': str(e),
            'type': 'finetuned_encoder'
        }
        clear_memory()

print('\nFine-tuned encoder baselines done.')

In [ ]:
def maritaca_invoke(prompt, max_tokens=20):
    url = 'https://chat.maritaca.ai/api/chat/completions'
    headers = {
        'Authorization': f'Key {MARITACA_KEY}',
        'Content-Type': 'application/json'
    }
    body = {
        'model': MARITACA_MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'temperature': 0.0
    }

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(url, headers=headers, json=body, timeout=40)
            r.raise_for_status()
            return r.json()['choices'][0]['message']['content'].strip()
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(API_SLEEP * (attempt + 2))
            else:
                logger.error(f'Maritaca error: {e}')
                return 'ERROR'

def run_llm_eval(model_key, invoke_fn, label='LLM', flush_every=1):
    ckpt = load_or_init_ckpt(model_key)
    logger.info(f'Starting eval: {model_key}')

    cls_done = set(ckpt.get('cls', {}).keys())
    cls_todo = [inst for inst in instances_cls if inst['jacv_id'] not in cls_done]

    if cls_todo:
        print(f'{model_key} | CLS remaining: {len(cls_todo)}')

    for i, inst in enumerate(tqdm(cls_todo, desc=f'{label} CLS'), 1):
        p = prompt_cls(inst['direito'], inst['pedido'])
        resp = invoke_fn(p)
        pred = parse_cls(resp)

        ckpt['cls'][inst['jacv_id']] = {
            'gold': inst['gold_label'],
            'pred': pred,
            'raw': resp
        }

        if (i % flush_every == 0) or (i == len(cls_todo)):
            flush_ckpt(model_key, ckpt)

        time.sleep(API_SLEEP)

    adv_done = set(ckpt.get('adv', {}).keys())
    adv_todo = [inst for inst in instances_adv if inst['jacv_id'] not in adv_done]

    if adv_todo:
        print(f'{model_key} | ADV remaining: {len(adv_todo)}')

    for i, inst in enumerate(tqdm(adv_todo, desc=f'{label} ADV'), 1):
        p = prompt_adv(inst['direito'], inst['pedido_A'], inst['pedido_B'])
        resp = invoke_fn(p)
        pred = parse_adv(resp)

        ckpt['adv'][inst['jacv_id']] = {
            'gold': inst['coherent_option'],
            'pred': pred,
            'raw': resp
        }

        if (i % flush_every == 0) or (i == len(adv_todo)):
            flush_ckpt(model_key, ckpt)

        time.sleep(API_SLEEP)

    result = finalize_result_from_ckpt(model_key, 'llm')
    return result

RESULTS['models']['Sabia-4'] = run_llm_eval('Sabia-4', maritaca_invoke, 'Sabiá-4', flush_every=1)
print(f'Sabiá-4 done: CLS={RESULTS["models"]["Sabia-4"]["cls_acc"]:.3f} ADV={RESULTS["models"]["Sabia-4"]["adv_acc"]:.3f}')

In [ ]:
def bedrock_invoke(model_id, prompt, max_tokens=20):
    family = (
        'anthropic' if 'anthropic.' in model_id else
        'meta' if 'meta.' in model_id else
        'mistral' if 'mistral.' in model_id else
        'other'
    )

    if family == 'anthropic':
        body = json.dumps({
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': max_tokens,
            'messages': [{'role': 'user', 'content': prompt}]
        })
    elif family == 'meta':
        fp = f'<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{prompt}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>'
        body = json.dumps({'prompt': fp, 'max_gen_len': max_tokens, 'temperature': 0.0})
    elif family == 'mistral':
        body = json.dumps({
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': max_tokens,
            'temperature': 0.0
        })
    else:
        body = json.dumps({
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': max_tokens
        })

    for attempt in range(MAX_RETRIES):
        try:
            resp = bedrock.invoke_model(modelId=model_id, body=body)
            res = json.loads(resp['body'].read())

            if family == 'anthropic':
                text = res['content'][0]['text']
            elif family == 'meta':
                text = res.get('generation', '')
            elif family == 'mistral':
                text = res['choices'][0]['message']['content']
            else:
                text = res.get('choices', [{}])[0].get('message', {}).get('content', '')

            return str(text).strip()

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(API_SLEEP * (attempt + 2))
            else:
                logger.error(f'Bedrock {model_id} error: {e}')
                return 'ERROR'

for friendly, model_id in BEDROCK_MODELS.items():
    def _invoke(prompt, mid=model_id):
        return bedrock_invoke(mid, prompt)

    RESULTS['models'][friendly] = run_llm_eval(friendly, _invoke, friendly)
    r = RESULTS['models'][friendly]
    print(f'{friendly}: CLS={r["cls_acc"]:.3f} ADV={r["adv_acc"]:.3f}')

In [ ]:
df = pd.DataFrame({k: v for k, v in RESULTS['models'].items()}).T

for col in ['cls_acc', 'cls_f1_macro', 'adv_acc', 'adv_acc_answered_only']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('adv_acc', ascending=False)

print('=' * 72)
print('  FINAL PERFORMANCE REPORT — JACV EVALUATION')
print('=' * 72)
cols_show = [c for c in ['type', 'cls_acc', 'cls_f1_macro', 'adv_acc', 'adv_acc_answered_only'] if c in df.columns]
print(df[cols_show].to_string())
print('=' * 72)

RESULTS['metadata']['completed_at'] = datetime.now().isoformat()
with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=str)

logger.info(f'Results saved: {RESULTS_JSON}')
print(f'Saved -> {RESULTS_JSON}')

In [ ]:
df_plot = df.dropna(subset=['cls_acc', 'adv_acc']).copy()

colors = {
    'statistical': '#95A5A6',
    'embedding_supervised': '#2980B9',
    'finetuned_encoder': '#8E44AD',
    'llm': '#E74C3C',
    'llm_cot': '#E67E22'
}
bar_colors = [colors.get(str(t), '#888888') for t in df_plot['type']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, title in [
    (axes[0], 'cls_acc', 'JACV-CLS (3-class Accuracy)'),
    (axes[1], 'adv_acc', 'JACV-ADV (Adversarial Accuracy)')
]:
    vals = df_plot[col].values
    bars = ax.barh(df_plot.index, vals, color=bar_colors, alpha=0.88, edgecolor='white')

    ax.axvline(0.333, color='gray', linestyle='--', alpha=0.5, label='Random (CLS)')
    ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance (ADV)')

    for bar, v in zip(bars, vals):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9, fontweight='bold')

    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Accuracy')

import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items() if l in df_plot['type'].unique()]
fig.legend(handles=legend_patches, loc='lower center', ncol=min(5, len(legend_patches)), bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
p = FIGURES_DIR / 'figM01_cls_adv_grouped_bar.png'
fig.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved -> {p}')

In [ ]:
metrics_cols = ['cls_acc', 'cls_f1_macro', 'adv_acc']
df_radar = df_plot.dropna(subset=metrics_cols).copy()

if not df_radar.empty:
    model_names = list(df_radar.index)
    N = len(metrics_cols)
    angles = [n / float(N) * 2 * pi for n in range(N)] + [0]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['CLS Accuracy', 'CLS F1-Macro', 'ADV Accuracy'], size=11)
    ax.set_ylim(0, 1)
    plt.yticks([0.25, 0.5, 0.75, 1.0], ['25%', '50%', '75%', '100%'], size=8)

    cmap = plt.cm.get_cmap('tab10', len(model_names))
    for i, m in enumerate(model_names):
        vals = df_radar.loc[m, metrics_cols].tolist()
        vals = vals + [vals[0]]
        ax.plot(angles, vals, linewidth=2, label=m, color=cmap(i))
        ax.fill(angles, vals, alpha=0.07, color=cmap(i))

    plt.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=9)
    plt.title('Figure M02 — JACV Performance Radar\nAll Models', fontweight='bold', y=1.08)
    p = FIGURES_DIR / 'figM02_radar_chart.png'
    plt.savefig(p, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved -> {p}')

stat_models = df_plot[df_plot['type'] == 'statistical']['adv_acc']
embed_models = df_plot[df_plot['type'].isin(['embedding_supervised', 'finetuned_encoder'])]['adv_acc']
llm_models = df_plot[df_plot['type'] == 'llm']['adv_acc']

fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance (ADV)')

for i, (name, val) in enumerate(stat_models.items()):
    ax.scatter(i, val, s=120, color='#95A5A6', zorder=5, label='Statistical' if i == 0 else '')
    ax.text(i, val + 0.01, name, ha='center', fontsize=8, rotation=15)

offset = len(stat_models)
for i, (name, val) in enumerate(embed_models.items()):
    ax.scatter(i + offset, val, s=120, color='#8E44AD', zorder=5, marker='D', label='Encoder-based' if i == 0 else '')
    ax.text(i + offset, val + 0.01, name, ha='center', fontsize=8, rotation=15)

offset2 = offset + len(embed_models)
for i, (name, val) in enumerate(llm_models.items()):
    ax.scatter(i + offset2, val, s=150, color='#E74C3C', zorder=5, marker='*', label='LLM' if i == 0 else '')
    ax.text(i + offset2, val + 0.01, name, ha='center', fontsize=8, rotation=15)

ax.set_ylim(0, 1.1)
ax.set_ylabel('ADV Accuracy')
ax.set_title('Figure M03 — The Reasoning Gap: Statistical / Encoder / LLM\n(JACV-ADV Task)', fontweight='bold')
ax.legend()
ax.set_xticks([])
ax.set_xlabel('Models')
p = FIGURES_DIR / 'figM03_reasoning_gap.png'
fig.savefig(p, bbox_inches='tight')
plt.show()
print(f'Saved -> {p}')

In [ ]:
selected_models = []
for model_name, row in df.iterrows():
    if row.get('type') in ('llm', 'finetuned_encoder'):
        if ckpt_path(model_name).exists():
            selected_models.append(model_name)

f1_data = {}

for model_key in selected_models:
    ckpt = load_prediction_checkpoint(model_key)
    if ckpt is None or not ckpt.get('cls'):
        continue

    y = [v['gold'] for v in ckpt['cls'].values()]
    p = [v['pred'] for v in ckpt['cls'].values()]

    report = classification_report(
        y, p,
        output_dict=True,
        zero_division=0,
        labels=CLS_LABELS
    )
    f1_data[model_key] = {lbl: report.get(lbl, {}).get('f1-score', 0.0) for lbl in CLS_LABELS}

if f1_data:
    df_f1 = pd.DataFrame(f1_data).T
    df_f1 = df_f1.loc[df.loc[df_f1.index].sort_values('adv_acc', ascending=False).index]

    fig, ax = plt.subplots(figsize=(8, max(3, len(df_f1) * 0.55)))
    sns.heatmap(
        df_f1,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        ax=ax,
        vmin=0,
        vmax=1,
        linewidths=0.5,
        cbar_kws={'label': 'F1-Score'}
    )
    ax.set_title('Figure M04 — Per-Class F1 Score\n(LLMs + Fine-Tuned Encoders)', fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Model')

    p = FIGURES_DIR / 'figM04_perclass_f1_heatmap.png'
    fig.savefig(p, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {p}')
else:
    print('No checkpoint-backed models available for per-class F1 heatmap.')

In [ ]:
def prompt_cot_adv(direito, pedido_a, pedido_b):
    return '\n'.join([
        'Tarefa: Avalie a coerência lógica entre a fundamentação jurídica e as duas opções de pedido.',
        'Regra: Se não houver clareza lógica suficiente, conclua como INCONCLUSIVO.',
        '',
        '--- FUNDAMENTAÇÃO JURÍDICA (DIREITO) ---',
        trunc(direito, 500),
        '--- PEDIDO A ---',
        trunc(pedido_a, 300),
        '--- PEDIDO B ---',
        trunc(pedido_b, 300),
        '---',
        'Analise passo-a-passo e forneça sua resposta estritamente no formato abaixo:',
        'Fatos Relevantes: [resumo]',
        'Norma Aplicada: [analise]',
        'Nexo Logico: [reflexao]',
        'Conclusao: [Escolha apenas A, B ou INCONCLUSIVO]'
    ])

def parse_cot_adv(text):
    t = str(text).upper().strip()

    if 'CONCLUS' in t:
        t = t.split('CONCLUS')[-1]

    if 'INCONCLUSIVO' in t:
        return 'INCONCLUSIVO'
    if ' A' in t[-30:] or '"A"' in t[-30:] or '\nA' in t[-30:]:
        return 'A'
    if ' B' in t[-30:] or '"B"' in t[-30:] or '\nB' in t[-30:]:
        return 'B'
    return 'ERROR'

def run_cot_eval(model_key, invoke_fn, flush_every=1):
    cot_key = f'{model_key}-CoT'
    ckpt = load_or_init_ckpt(cot_key)

    adv_done = set(ckpt.get('adv', {}).keys())
    adv_todo = [inst for inst in instances_adv if inst['jacv_id'] not in adv_done]

    if adv_todo:
        print(f'{cot_key} | ADV remaining: {len(adv_todo)}')

    for i, inst in enumerate(tqdm(adv_todo, desc=f'{cot_key}'), 1):
        p = prompt_cot_adv(inst['direito'], inst['pedido_A'], inst['pedido_B'])
        resp = invoke_fn(p)
        pred = parse_cot_adv(resp)

        ckpt['adv'][inst['jacv_id']] = {
            'gold': inst['coherent_option'],
            'pred': pred,
            'raw': resp
        }

        if (i % flush_every == 0) or (i == len(adv_todo)):
            flush_ckpt(cot_key, ckpt)

        time.sleep(API_SLEEP)

    p_a = [v['pred'] for v in ckpt['adv'].values()]
    y_a = [v['gold'] for v in ckpt['adv'].values()]

    raw_acc = accuracy_score(y_a, p_a)

    answered_idx = [i for i, pred in enumerate(p_a) if pred in ('A', 'B')]
    answered_acc = accuracy_score(
        [y_a[i] for i in answered_idx],
        [p_a[i] for i in answered_idx]
    ) if answered_idx else np.nan

    return {
        'cls_acc': np.nan,
        'cls_f1_macro': np.nan,
        'adv_acc': float(raw_acc),
        'adv_acc_answered_only': float(answered_acc) if not np.isnan(answered_acc) else np.nan,
        'adv_errors': p_a.count('ERROR'),
        'inconclusive': p_a.count('INCONCLUSIVO'),
        'adv_acc_ci95': bootstrap_ci_accuracy(y_a, p_a),
        'type': 'llm_cot'
    }

RESULTS['models']['Sabia-4-CoT'] = run_cot_eval('Sabia-4', maritaca_invoke, flush_every=1)

def claude_sonnet_46_func(prompt):
    return bedrock_invoke('global.anthropic.claude-sonnet-4-6', prompt, 400)

RESULTS['models']['Claude-Sonnet-4.6-CoT'] = run_cot_eval('Claude-Sonnet-4.6', claude_sonnet_46_func, flush_every=1)

def mistral_large_3_func(prompt):
    return bedrock_invoke('mistral.mistral-large-3-675b-instruct', prompt, 400)

RESULTS['models']['Mistral-Large-3-CoT'] = run_cot_eval('Mistral-Large-3', mistral_large_3_func, flush_every=1)

print('\nLegal-CoT models done.')
for key in ['Sabia-4-CoT', 'Claude-Sonnet-4.6-CoT', 'Mistral-Large-3-CoT']:
    r = RESULTS['models'][key]
    print(
        f'{key}: ADV_RAW={r["adv_acc"]:.3f} | '
        f'ADV_ANSWERED={r["adv_acc_answered_only"]:.3f} | '
        f'INCONCLUSIVE={r["inconclusive"]} | ERROR={r["adv_errors"]}'
    )

In [ ]:
df_final = pd.DataFrame({k: v for k, v in RESULTS['models'].items()}).T

for col in ['cls_acc', 'cls_f1_macro', 'adv_acc', 'adv_acc_answered_only']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

df_final = df_final.sort_values('adv_acc', ascending=False)

RESULTS['metadata']['completed_at'] = datetime.now().isoformat()

with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=str)

csv_path = TABLES_DIR / 'publication_model_metrics.csv'
tex_path = TABLES_DIR / 'publication_model_metrics.tex'

df_final.to_csv(csv_path, encoding='utf-8-sig')

latex_cols = [c for c in ['type', 'cls_acc', 'cls_f1_macro', 'adv_acc', 'adv_acc_answered_only'] if c in df_final.columns]
with open(tex_path, 'w', encoding='utf-8') as f:
    f.write(df_final[latex_cols].to_latex(float_format='%.4f'))

print(f'Saved JSON -> {RESULTS_JSON}')
print(f'Saved CSV  -> {csv_path}')
print(f'Saved TEX  -> {tex_path}')

print('\n=== FINAL TABLE ===')
print(df_final[latex_cols].to_string())